# Results Summary — Test Set

Consolidated comparison of all configurations on the **test split** (baseline vs every regularization and the higher-resolution run). Everything is reported on test only, as mean ± standard error over the available seeds.

1. Overall test metrics per configuration (mAP50-95, dice, precision, recall)
2. Per-class test dice per configuration
3. Summary table

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

CLASSES = ['bicycle', 'bus', 'car', 'motorcycle', 'person', 'rider', 'train', 'truck']
CONFIG_ORDER = ['baseline', 'weight decay', 'L2 + LS', 'augmentation', 'higher res']
CONFIG_COLORS = {'baseline': '#999999', 'weight decay': '#C73E1D', 'L2 + LS': '#9B59B6',
                 'augmentation': '#6A994E', 'higher res': '#2E86AB'}
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.3})

df = pd.read_csv(Path('..') / 'all_results.csv')
if 'State' in df.columns:
    df = df[df['State'] == 'finished']
df = df.copy()
for c in df.columns:
    if c.startswith(('test/', 'class/')) or c in ('weight_decay', 'seed'):
        df[c] = pd.to_numeric(df[c], errors='coerce')

def assign(row):
    g, n, wd = str(row.get('Group', '')), str(row['Name']), row.get('weight_decay')
    if g == '4.2 baseline':
        return 'baseline'
    if g == '4.4 augmentation':
        return 'augmentation'
    if g == '4.5 img size':
        return 'higher res'
    if n == 'yolo-seg-reg-l2-ls':
        return 'L2 + LS'
    if pd.notna(wd) and abs(float(wd) - 0.001) < 1e-9:
        return 'weight decay'
    return None

df['config'] = df.apply(assign, axis=1)
sub = df[df['config'].notna()].copy()
present = [c for c in CONFIG_ORDER if c in sub['config'].unique()]

print('configs and seeds:')
for c in present:
    seeds = sorted(sub[sub['config'] == c]['seed'].dropna().astype(int).tolist())
    print(f'  {c:14s} n={len(seeds)}  seeds={seeds}')

## 1. Overall test metrics per configuration

Mean ± SE over seeds. Note that L2 + LS has a single seed, so it has no error bar.

In [ ]:
metrics = ['mask_mAP50_95', 'mask_dice', 'mask_precision', 'mask_recall']
labels = ['mAP50-95', 'dice', 'precision', 'recall']

x = np.arange(len(metrics)); n = len(present); w = 0.8 / n
fig, ax = plt.subplots(figsize=(12, 5))
for i, cfg in enumerate(present):
    grp = sub[sub['config'] == cfg]
    m = [grp[f'test/{mt}'].mean() for mt in metrics]
    e = [np.nan_to_num(grp[f'test/{mt}'].sem()) for mt in metrics]
    ax.bar(x + (i - (n - 1) / 2) * w, m, w, yerr=e, capsize=3,
           color=CONFIG_COLORS[cfg], alpha=0.85, label=cfg)
ax.set_xticks(x); ax.set_xticklabels(labels)
ax.set_ylabel('test mask metric')
ax.set_title('Test metrics by configuration (mean ± SE over seeds)')
ax.legend(ncol=len(present), fontsize=8); plt.tight_layout(); plt.show()

## 2. Per-class test dice per configuration

Heat map of the per-class test dice, averaged over seeds. Rows are configurations, columns are classes. This shows where each configuration helps and which classes stay weak.

In [ ]:
mat = np.array([[sub[sub['config'] == cfg][f'class/test_mask_dice/{cls}'].mean() for cls in CLASSES]
                for cfg in present])

fig, ax = plt.subplots(figsize=(11, 4.5))
im = ax.imshow(mat, aspect='auto', cmap='YlGn')
ax.set_xticks(range(len(CLASSES))); ax.set_xticklabels(CLASSES, rotation=30, ha='right')
ax.set_yticks(range(len(present))); ax.set_yticklabels(present)
for i in range(len(present)):
    for j in range(len(CLASSES)):
        ax.text(j, i, f'{mat[i, j]:.2f}', ha='center', va='center', fontsize=8)
fig.colorbar(im, ax=ax, label='test mask dice')
ax.set_title('Per-class test dice by configuration')
ax.grid(False); plt.tight_layout(); plt.show()

## 3. Summary table

Test metrics per configuration as mean ± SE, with the improvement over the baseline on the primary metric (mask mAP50-95).

In [ ]:
base_map = sub[sub['config'] == 'baseline']['test/mask_mAP50_95'].mean()
rows = []
for cfg in present:
    grp = sub[sub['config'] == cfg]
    row = {'config': cfg, 'n': len(grp)}
    for mt, lb in zip(metrics, labels):
        row[lb] = f"{grp[f'test/{mt}'].mean():.3f} \u00b1 {np.nan_to_num(grp[f'test/{mt}'].sem()):.3f}"
    row['\u0394mAP vs base'] = f"{grp['test/mask_mAP50_95'].mean() - base_map:+.3f}"
    rows.append(row)
print(pd.DataFrame(rows).to_string(index=False))